# Домашняя работа 5 Верстов Родион

In [6]:
from datetime import datetime
from typing import List, Dict, Union, Optional
import json

class Account:
 
    def __init__(self, account_holder: str, balance: float = 0):

        # Проверка на отрицательный начальный баланс
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        
        self.holder = account_holder
        self._balance = balance
        self.operations_history = []  # История операций
        
        # Добавляем начальную операцию создания счета
        self._add_operation(
            operation_type='init',
            amount=balance,
            status='success',
            note=f"Счет создан с начальным балансом {balance}"
        )
    
    def _add_operation(self, operation_type: str, amount: float, status: str, 
                      note: str = "") -> None:

        operation = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
            'type': operation_type,
            'amount': amount,
            'balance_after': self._balance,
            'status': status,
            'note': note
        }
        self.operations_history.append(operation)
    
    def deposit(self, amount: float) -> bool:

        # Проверка на положительную сумму
        if amount <= 0:
            self._add_operation(
                operation_type='deposit',
                amount=amount,
                status='fail',
                note="Ошибка: сумма пополнения должна быть положительной"
            )
            print(f"Ошибка: сумма пополнения ({amount}) должна быть положительной")
            return False
        
        # Успешное пополнение
        self._balance += amount
        self._add_operation(
            operation_type='deposit',
            amount=amount,
            status='success',
            note=f"Успешное пополнение на {amount}"
        )
        print(f"Счет пополнен на {amount}. Текущий баланс: {self._balance}")
        return True
    
    def withdraw(self, amount: float) -> bool:
 
        # Проверка на положительную сумму
        if amount <= 0:
            self._add_operation(
                operation_type='withdraw',
                amount=amount,
                status='fail',
                note="Ошибка: сумма снятия должна быть положительной"
            )
            print(f"Ошибка: сумма снятия ({amount}) должна быть положительной")
            return False
        
        # Проверка на достаточность средств
        if amount > self._balance:
            self._add_operation(
                operation_type='withdraw',
                amount=amount,
                status='fail',
                note=f"Ошибка: недостаточно средств. Доступно: {self._balance}"
            )
            print(f"Ошибка: недостаточно средств. Доступно: {self._balance}")
            return False
        
        # Успешное снятие
        self._balance -= amount
        self._add_operation(
            operation_type='withdraw',
            amount=amount,
            status='success',
            note=f"Успешное снятие {amount}"
        )
        print(f"Снято {amount}. Текущий баланс: {self._balance}")
        return True
    
    def get_balance(self) -> float:
    
        return self._balance
    
    def get_history(self, pretty_print: bool = False) -> List[Dict]:
     
        if pretty_print:
            print(f"\nИстория операций по счету '{self.holder}':")
            print("-" * 80)
            for i, op in enumerate(self.operations_history, 1):
                status_symbol = "+" if op['status'] == 'success' else "-"
                op_type_rus = {
                    'init': 'Создание',
                    'deposit': 'Пополнение',
                    'withdraw': 'Снятие'
                }.get(op['type'], op['type'])
                
                print(f"{i}. {status_symbol} {op['timestamp']}")
                print(f"   Тип: {op_type_rus}")
                print(f"   Сумма: {op['amount']}")
                print(f"   Баланс после: {op['balance_after']}")
                if op['note']:
                    print(f"   Примечание: {op['note']}")
                print()
        
        return self.operations_history
    
    def __str__(self) -> str:
        
        return f"Account(holder='{self.holder}', balance={self._balance})"
    
    def __repr__(self) -> str:

        return f"Account('{self.holder}', {self._balance})"

In [7]:
class CreditAccount(Account):
    
    def __init__(self, account_holder: str, balance: float = 0, credit_limit: float = 1000):

        # Для кредитного счета баланс может быть отрицательным,
        # но не ниже -credit_limit
        if balance < -credit_limit:
            raise ValueError(f"Баланс не может быть меньше -{credit_limit}")
        
        # Вызываем конструктор родительского класса
        super().__init__(account_holder, max(0, balance))
        
        self.credit_limit = credit_limit
        
        # Если начальный баланс отрицательный, корректируем его
        if balance < 0:
            self._balance = balance  # Присваиваем напрямую, т.к. проверка уже была
            # Обновляем запись о создании счета
            self.operations_history[-1]['balance_after'] = balance
            self.operations_history[-1]['note'] = f"Кредитный счет создан с балансом {balance}, лимит {credit_limit}"
    
    def _add_operation(self, operation_type: str, amount: float, status: str, 
                      note: str = "", credit_used: Optional[float] = None) -> None:
  
        operation = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
            'type': operation_type,
            'amount': amount,
            'balance_after': self._balance,
            'status': status,
            'note': note
        }
        
        # Добавляем информацию о кредите, если она есть
        if credit_used is not None:
            operation['credit_used'] = credit_used
            operation['own_funds_used'] = amount - credit_used
        
        self.operations_history.append(operation)
    
    def withdraw(self, amount: float) -> bool:
    
        # Проверка на положительную сумму
        if amount <= 0:
            self._add_operation(
                operation_type='withdraw',
                amount=amount,
                status='fail',
                note="Ошибка: сумма снятия должна быть положительной"
            )
            print(f"Ошибка: сумма снятия ({amount}) должна быть положительной")
            return False
        
        # Проверка на доступность средств с учетом кредитного лимита
        available_funds = self._balance + self.credit_limit
        if amount > available_funds:
            self._add_operation(
                operation_type='withdraw',
                amount=amount,
                status='fail',
                note=f"Ошибка: превышение кредитного лимита. Доступно: {available_funds}"
            )
            print(f"Ошибка: превышение кредитного лимита. Доступно: {available_funds}")
            return False
        
        # Расчет использованных кредитных средств
        old_balance = self._balance
        credit_used = 0
        
        if self._balance >= amount:
            # Снимаем только свои средства
            credit_used = 0
        else:
            # Используем кредитные средства
            credit_used = amount - max(0, self._balance)
        
        # Успешное снятие
        self._balance -= amount
        
        # Формируем примечание
        if credit_used > 0:
            own_used = amount - credit_used
            note = f"Снятие: своих {own_used}, кредитных {credit_used}"
        else:
            note = f"Снятие только своих средств: {amount}"
        
        self._add_operation(
            operation_type='withdraw',
            amount=amount,
            status='success',
            note=note,
            credit_used=credit_used
        )
        
        print(f"Снято {amount}. Текущий баланс: {self._balance}")
        if credit_used > 0:
            print(f"Использовано кредитных средств: {credit_used}")
        print(f"Доступный кредитный лимит: {self.get_available_credit()}")
        
        return True
    
    def get_available_credit(self) -> float:
   
        if self._balance >= 0:
            return self.credit_limit
        else:
            return self.credit_limit + self._balance  # _balance отрицательный
    
    def get_credit_usage(self) -> Dict:
  
        return {
            'credit_limit': self.credit_limit,
            'current_balance': self._balance,
            'available_credit': self.get_available_credit(),
            'credit_used': max(0, -self._balance) if self._balance < 0 else 0,
            'own_funds': max(0, self._balance)
        }
    
    def __str__(self) -> str:
        
        return (f"CreditAccount(holder='{self.holder}', balance={self._balance}, "
                f"credit_limit={self.credit_limit})")

In [8]:
def demonstrate_account():
    print("\n" + "=" * 60)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ ОБЫЧНОГО СЧЕТА")
    print("=" * 60)
    
    # Создание счета
    print("\n1. Создание счета:")
    acc = Account("Иван Петров", 1000)
    print(acc)
    
    # Пополнение счета
    print("\n2. Пополнение счета:")
    acc.deposit(500)
    acc.deposit(-100)  # Попытка пополнения с отрицательной суммой
    
    # Снятие средств
    print("\n3. Снятие средств:")
    acc.withdraw(300)
    acc.withdraw(1500)  # Попытка снятия больше, чем на счете
    
    # Текущий баланс
    print(f"\nТекущий баланс: {acc.get_balance()}")
    
    # История операций
    print("\n4. История операций:")
    acc.get_history(pretty_print=True)


def demonstrate_credit_account():
    print("\n" + "=" * 60)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ КРЕДИТНОГО СЧЕТА")
    print("=" * 60)
    
    # Создание кредитного счета
    print("\n1. Создание кредитного счета с лимитом 2000:")
    credit_acc = CreditAccount("Анна Сидорова", 500, credit_limit=2000)
    print(credit_acc)
    print(f"Информация о кредите: {credit_acc.get_credit_usage()}")
    
    # Снятие средств (в пределах своих средств)
    print("\n2. Снятие в пределах своих средств (300):")
    credit_acc.withdraw(300)
    
    # Снятие с использованием кредита
    print("\n3. Снятие с использованием кредита (1000):")
    credit_acc.withdraw(1000)
    
    # Попытка превысить кредитный лимит
    print("\n4. Попытка превысить кредитный лимит (1500):")
    credit_acc.withdraw(1500)
    
    # Проверка доступного кредита
    print(f"\nТекущий баланс: {credit_acc.get_balance()}")
    print(f"Доступный кредит: {credit_acc.get_available_credit()}")
    print(f"Полная информация: {credit_acc.get_credit_usage()}")
    
    # Пополнение счета (погашение кредита)
    print("\n5. Пополнение счета (погашение кредита):")
    credit_acc.deposit(800)
    
    # История операций кредитного счета
    print("\n6. История операций кредитного счета:")
    credit_acc.get_history(pretty_print=True)


def demonstrate_edge_cases():
    print("\n" + "=" * 60)
    print("ДЕМОНСТРАЦИЯ ГРАНИЧНЫХ СЛУЧАЕВ")
    print("=" * 60)
    
    # Попытка создания с отрицательным балансом
    print("\n1. Попытка создания обычного счета с отрицательным балансом:")
    try:
        acc = Account("Тест", -100)
    except ValueError as e:
        print(f"Ошибка: {e}")
    
    # Создание кредитного счета с отрицательным балансом в пределах лимита
    print("\n2. Создание кредитного счета с отрицательным балансом:")
    credit_acc = CreditAccount("Тест Кредит", -500, credit_limit=1000)
    print(credit_acc)
    
    # Попытка создания кредитного счета с балансом ниже лимита
    print("\n3. Попытка создания кредитного счета с балансом ниже лимита:")
    try:
        credit_acc = CreditAccount("Тест Кредит", -1500, credit_limit=1000)
    except ValueError as e:
        print(f"Ошибка: {e}")
    
    # Работа с нулевым балансом
    print("\n4. Работа с нулевым балансом:")
    acc_zero = CreditAccount("Нулевой", 0, credit_limit=500)
    print(f"Начальный баланс: {acc_zero.get_balance()}")
    print(f"Доступный кредит: {acc_zero.get_available_credit()}")
    acc_zero.withdraw(300)  # Снятие в кредит


In [9]:
if __name__ == "__main__":
   
    demonstrate_account()
    demonstrate_credit_account()
    demonstrate_edge_cases()

ТЕСТИРОВАНИЕ БАНКОВСКИХ СЧЕТОВ

ДЕМОНСТРАЦИЯ РАБОТЫ ОБЫЧНОГО СЧЕТА

1. Создание счета:
Account(holder='Иван Петров', balance=1000)

2. Пополнение счета:
Счет пополнен на 500. Текущий баланс: 1500
Ошибка: сумма пополнения (-100) должна быть положительной

3. Снятие средств:
Снято 300. Текущий баланс: 1200
Ошибка: недостаточно средств. Доступно: 1200

Текущий баланс: 1200

4. История операций:

История операций по счету 'Иван Петров':
--------------------------------------------------------------------------------
1. + 2026-03-08 22:38:30.443
   Тип: Создание
   Сумма: 1000
   Баланс после: 1000
   Примечание: Счет создан с начальным балансом 1000

2. + 2026-03-08 22:38:30.443
   Тип: Пополнение
   Сумма: 500
   Баланс после: 1500
   Примечание: Успешное пополнение на 500

3. - 2026-03-08 22:38:30.443
   Тип: Пополнение
   Сумма: -100
   Баланс после: 1500
   Примечание: Ошибка: сумма пополнения должна быть положительной

4. + 2026-03-08 22:38:30.443
   Тип: Снятие
   Сумма: 300
   Балан